# ◆ Corax LM — обучение умной модели (medium) в бесплатном Google Colab

Здесь модель учится **с нуля** на видеокарте Colab, а потом ты запускаешь её на телефоне в Termux.

1. **Среда выполнения → Сменить среду выполнения → T4 GPU** (бесплатно).
2. Запускай ячейки по порядку. Прогресс сохраняется на Google Диск: если Colab отключится, запусти всё заново — обучение продолжится с места остановки.
3. В конце скачай `chat-q8.bin` (или `chat-q4.bin`) и положи на телефон в `ai/models/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/corax

## 1. Код
Если репозиторий приватный, загрузи папку `ai` архивом и распакуй в `/content/ai`.

In [ ]:
import os
if not os.path.exists('/content/ai'):
    !git clone --depth 1 https://github.com/CheatRBX/GTA-6-MP.git /content/repo && cp -r /content/repo/ai /content/ai
%cd /content/ai
!make -s && ./corax presets

## 2. Данные: русская Википедия + рассказы

Один файл Википедии — это примерно 1,5 ГБ текста и ~200 млн токенов, на 2–3 часа обучения на T4. Хочешь умнее — бери больше файлов (`SHARDS`) и учи дольше.

In [ ]:
SHARDS = 1   # сколько файлов Википедии взять (всего 21)
import os, urllib.request, pandas as pd
os.makedirs('data/raw', exist_ok=True)
for i in range(SHARDS):
    url = f'https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/20231101.ru/train-{i:05d}-of-00021.parquet'
    pq = f'data/raw/wiki{i}.parquet'
    if not os.path.exists(f'data/raw/wiki{i}.txt'):
        urllib.request.urlretrieve(url, pq)
        df = pd.read_parquet(pq, columns=['text'])
        with open(f'data/raw/wiki{i}.txt', 'w', encoding='utf-8') as f:
            for t in df['text']:
                f.write(t.strip() + '\n<|endoftext|>\n')
        os.remove(pq)
        print('готово', i, len(df), 'статей')
!bash scripts/get_data.sh ru-stories
!du -sh data/raw

## 3. Свой токенизатор и подготовка (10–20 минут)

In [ ]:
import os
if not os.path.exists('/content/drive/MyDrive/corax/train.bin'):
    !python3 tools/prepare.py pretrain data/raw --out-dir data --vocab 16384 --min-cyr 0.5 --tok-sample-mb 50
    !cp data/tokenizer.bin data/train.bin data/val.bin /content/drive/MyDrive/corax/
else:
    !cp /content/drive/MyDrive/corax/tokenizer.bin /content/drive/MyDrive/corax/train.bin /content/drive/MyDrive/corax/val.bin data/

## 4. Обучение с нуля — medium (~88 млн параметров)

`STEPS=12000` при батче 32×1024 — это ~390 млн токенов (на T4 несколько часов). Если Colab отключился, просто запусти ячейку снова: сработает `--resume`.

In [ ]:
STEPS = 12000
import os
out = '/content/drive/MyDrive/corax/base.bin'
resume = '--resume' if os.path.exists(out) else ''
!python3 tools/train_torch.py --preset medium --data data/train.bin --val data/val.bin \
    --tokenizer data/tokenizer.bin --out {out} --steps {STEPS} --batch 16 --accum 2 \
    --save-every 500 --eval-every 500 --compile {resume}

In [ ]:
# Проверка: продолжение текста базовой моделью
!./corax gen -m /content/drive/MyDrive/corax/base.bin -p "Москва — это" -n 80 --temp 0.7

## 5. Дообучение на диалогах (ассистент)

Добавь свои диалоги в `data/my_dialogs.jsonl` (формат описан в README) — чем больше, тем лучше.

In [ ]:
import glob
files = ' '.join(['data/sft_example.jsonl'] + glob.glob('data/my_*.jsonl'))
!python3 tools/prepare.py sft {files} --tokenizer data/tokenizer.bin --out data/sft --repeat 3
!python3 tools/train_torch.py --init /content/drive/MyDrive/corax/base.bin --data data/sft \
    --out /content/drive/MyDrive/corax/chat.bin --steps 600 --batch 16 --lr 1e-4 --save-every 200

## 6. Квантование и скачивание

In [ ]:
!./corax quantize /content/drive/MyDrive/corax/chat.bin /content/drive/MyDrive/corax/chat-q8.bin q8
!./corax quantize /content/drive/MyDrive/corax/chat.bin /content/drive/MyDrive/corax/chat-q4.bin q4
!printf 'Привет! Кто ты?\n' | ./corax chat -m /content/drive/MyDrive/corax/chat-q8.bin --no-bar
from google.colab import files
files.download('/content/drive/MyDrive/corax/chat-q8.bin')

На телефоне:
```bash
cd ~/GTA-6-MP/ai
mkdir -p models && cp ~/storage/downloads/chat-q8.bin models/   # после termux-setup-storage
./corax bench -m models/chat-q8.bin --sweep
./corax chat  -m models/chat-q8.bin
```
Чтобы дообучать на телефоне дальше, скачай и f32-файл `chat.bin`: дообучение работает только с ним.